# Project 2: Viewing Experience Analysis

## Data Audit

This notebook inspects the final DuckDB warehouse created in Project 1 and identifies the fields available for analysis.

In [1]:
# Connect to the DuckDB warehouse created in Project 1

import duckdb

db_path = "../../data/warehouse/movie_experience.duckdb"

con = duckdb.connect(db_path)

print("Connected to warehouse.")

Connected to warehouse.


In [2]:
# List the tables available in the warehouse

con.sql("SHOW TABLES").df()

,name
0,analysis_dataset
1,experience_scores
2,titles


In [3]:
# Check the columns and data types in the titles table

con.sql("DESCRIBE titles").df()

,column_name,column_type,null,key,default,extra
0,title_key,VARCHAR,YES,None,None,None
1,tmdb_id,BIGINT,YES,None,None,None
2,media_type,VARCHAR,YES,None,None,None
3,title,VARCHAR,YES,None,None,None
4,original_title,VARCHAR,YES,None,None,None
5,overview,VARCHAR,YES,None,None,None
6,genres,VARCHAR[],YES,None,None,None
7,original_language,VARCHAR,YES,None,None,None
8,release_date,DATE,YES,None,None,None
9,runtime,INTEGER,YES,None,None,None


In [4]:
# Check the columns and data types in the experience scores table

con.sql("DESCRIBE experience_scores").df()

,column_name,column_type,null,key,default,extra
0,title_key,VARCHAR,YES,None,None,None
1,tmdb_id,BIGINT,YES,None,None,None
2,media_type,VARCHAR,YES,None,None,None
3,experience,VARCHAR,YES,None,None,None
4,category,VARCHAR,YES,None,None,None
5,score,DOUBLE,YES,None,None,None
6,raw_similarity,DOUBLE,YES,None,None,None
7,reviews_used,BIGINT,YES,None,None,None


In [5]:
# Check table size, uniqueness, and experience coverage before analysis

audit = con.sql("""
    SELECT
        (SELECT COUNT(*) FROM titles) AS title_rows,
        (SELECT COUNT(DISTINCT title_key) FROM titles) AS unique_titles,
        (SELECT COUNT(*) FROM experience_scores) AS score_rows,
        (SELECT COUNT(DISTINCT title_key) FROM experience_scores) AS titles_with_scores,
        (SELECT COUNT(DISTINCT experience) FROM experience_scores) AS experiences,
        (SELECT COUNT(DISTINCT category) FROM experience_scores) AS categories
""").df()

audit

,title_rows,unique_titles,score_rows,titles_with_scores,experiences,categories
0,7721,7721,424655,7721,55,10


In [6]:
# Create the main analytical view from the saved SQL file

sql_path = "../sql/01_analysis_dataset.sql"

with open(sql_path, "r") as file:
    con.execute(file.read())

print("Analysis dataset created.")

Analysis dataset created.


In [7]:
# Make sure the joined dataset has the expected number of rows

con.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT title_key) AS titles,
        COUNT(DISTINCT experience) AS experiences
    FROM analysis_dataset
""").df()

,rows,titles,experiences
0,424655,7721,55


In [8]:
con.close()